# 🪐 Finding Kepler-8b
## A Live Exoplanet Discovery Using Real NASA Data

In this notebook we will download **real data** from NASA's Kepler Space Telescope and find an actual planet — **Kepler-8b** — crossing in front of its star.

**About Kepler-8b** — discovered in 2010, it's a Hot Jupiter about 1,160 light-years away. It's about the size of Jupiter but orbits its star every 3.5 Earth days. We expect to see a transit dip of about 0.93%.

We will follow the same 8 steps real astronomers use:

1. Install the tool — `lightkurve`  
2. Import the libraries  
3. Search NASA's archive for data  
4. Download the data  
5. Plot the raw light curve  
6. Clean the data  
7. Fold the light curve at the known orbital period  
8. Measure the transit and calculate the planet's size  

Each step is one cell. Run them in order. The download in Step 4 takes ~30 seconds.

---

## Step 1 — Install `lightkurve`

`lightkurve` is the official Python library NASA scientists use to work with Kepler and TESS data.
The `!pip` command installs it inside the notebook. The `--quiet` flag just keeps the output short.
This only needs to be run **once per Colab session**.

In [ ]:
!pip install lightkurve --quiet
print("✅ lightkurve installed. Ready to find a planet!")

## Step 2 — Import the libraries we need

- `lightkurve` — the planet-hunting toolkit (we shorten it to `lk`)  
- `matplotlib` — for drawing the graphs  
- `numpy` — for the math  

We also configure matplotlib to make plots large and easy to read on a projector.

In [ ]:
# Import the libraries
import lightkurve as lk           # NASA's planet-hunting toolkit
import matplotlib.pyplot as plt   # Plotting library
import numpy as np                # Math/array library

# Silence non-critical warnings so the output stays clean
import warnings
warnings.filterwarnings('ignore')

# Make every plot big and projector-friendly
plt.rcParams['figure.figsize'] = (14, 6)   # Wide plots
plt.rcParams['font.size'] = 13              # Readable text
plt.rcParams['axes.grid'] = True            # Grid lines on
plt.rcParams['grid.alpha'] = 0.3            # Faint grid

print(f"✅ lightkurve version: {lk.__version__}")
print("🔭 Ready to look at the stars!")

## Step 3 — Search NASA's MAST archive for Kepler-8

MAST = **M**ikulski **A**rchive for **S**pace **T**elescopes — NASA's open data server that holds every observation Kepler, TESS, and JWST ever made.

We ask MAST: *"Show me every long-cadence light curve you have for the star Kepler-8."*

- `target` — the name of the star  
- `author="Kepler"` — we want the official Kepler mission data  
- `cadence="long"` — long cadence = one brightness measurement every 30 minutes  

It returns a table of all available data files.

In [ ]:
# Ask NASA's archive what data exists for Kepler-8
print("🔍 Searching MAST archive for Kepler-8 observations...")

search_result = lk.search_lightcurve("Kepler-8", author="Kepler", cadence="long")

print(f"\n📦 Found {len(search_result)} data files.")
print("    Each file = one quarter (~3 months) of Kepler observations.")

# Show the search results table
search_result

## Step 4 — Download one quarter of data

We will download just the **first file** — that's enough to find transits, and avoids waiting for a huge dataset.

- `search_result[0]` — pick the first file from the search results  
- `.download()` — fetch it from NASA's server to this notebook  

This takes about 20–30 seconds. While you wait, that's actual data traveling from a NASA server somewhere in Maryland to this notebook.

In [ ]:
# Download the first file from NASA
print("📡 Downloading data from NASA's server... (~30 sec)")

lc = search_result[0].download()   # 'lc' = light curve

# Tell us what we got
print(f"\n✅ Got {len(lc)} brightness measurements")
print(f"📅 Observation span: {lc.time.value[-1] - lc.time.value[0]:.1f} days")
print(f"⏱️  One measurement every ~30 minutes")

## Step 5 — Plot the raw light curve

Time to **see the data**. Each dot on this graph is one brightness measurement of the star Kepler-8.

- X-axis: time, in days  
- Y-axis: brightness, in electrons per second (how Kepler measures starlight)  

If a planet exists and crosses in front of the star, we should see **regular downward dips** at the same depth, spaced evenly in time.

In [ ]:
# Plot every brightness measurement as a dot
ax = lc.scatter(color='steelblue', s=4)

# Make the labels clear
ax.set_title("Kepler-8 — Raw brightness over 3 months\n"
             "(look for evenly-spaced downward dips!)", fontsize=14)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Brightness (electrons/sec)")

plt.tight_layout()
plt.show()

print("👀 See the vertical dips? Those are TRANSITS of planet Kepler-8b.")
print("   Each dip = one orbit. They should repeat every 3.52 days.")

## Step 6 — Clean the data

Kepler stares at stars for 3 months at a time, and during that time the star itself flickers slightly, and the spacecraft drifts a little. We need to remove these slow changes — without removing our precious transit signal.

We do **two** operations:

- `.normalize()` — divides every brightness value by the average, so the star's "normal" level becomes 1.0  
- `.flatten(window_length=401)` — smooths out slow variations using a 401-point sliding window. Crucially, this method has built-in sigma clipping that *protects the transits* while smoothing everything else.

> ⚠️ **We deliberately do NOT call** `remove_outliers()` here. Deep transits look like "outliers" to that function, and it would delete them. The `flatten()` method alone does the right thing.

In [ ]:
# Clean the data: normalize, then smooth out slow stellar variations
# flatten() preserves transits while removing long-term wobbles
lc_clean = lc.normalize().flatten(window_length=401)

# Plot the cleaned light curve
ax = lc_clean.scatter(color='darkblue', s=4)
ax.set_title("Kepler-8 — After cleaning. Transits stand out clearly.", fontsize=14)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Relative brightness  (1.0 = normal)")
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Normal brightness')
ax.legend()
plt.tight_layout()
plt.show()

# Sanity check — confirm the transits survived the cleaning
min_brightness = float(np.nanmin(lc_clean.flux.value))
print(f"📉 Each downward spike is a planet transit.")
print(f"   Lowest brightness reached: {min_brightness:.4f}")
print(f"   (Expected ~0.99 — a ~1% dip. If you see ~0.99, the transits survived!)")

## Step 7 — Fold the light curve

This is the magic trick. If we **already know** how often the planet orbits (its period), we can shift all the data so every transit lines up at the same place. Stacking many noisy transits creates one CLEAN transit signal.

**Values we use** (from published research on Kepler-8b):
- `PERIOD = 3.5224991 days` — the planet's year  
- `T0 = 121.1190` — the time (in Kepler days) when one transit happened  

The `.fold()` method takes care of the rest.

In [ ]:
# Kepler-8b's known orbital parameters (from NASA Exoplanet Archive)
PERIOD = 3.5224991  # days  -- the planet's "year"
T0     = 121.1190   # day of one known transit (in Kepler's time system)

# Fold the data: stack every transit on top of each other
folded = lc_clean.fold(period=PERIOD, epoch_time=T0)

# Plot the folded curve, zoomed in on the transit
ax = folded.scatter(color='steelblue', s=2, alpha=0.6)
ax.set_title(f"Kepler-8b — All transits stacked at period = {PERIOD} days", fontsize=14)
ax.set_xlabel("Time from transit center (days)")
ax.set_ylabel("Relative brightness")
ax.set_xlim(-0.3, 0.3)                     # zoom around the transit
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("🎯 One clean transit — built from every observation in the data!")

## Step 8 — Measure the transit and calculate the planet's size

Now the science. From the folded transit we can measure:

- **Depth** of the dip (how much the star dimmed)
- The relationship `depth = (R_planet / R_star)²` tells us the planet's size relative to its star

If we assume the star is Sun-sized (radius ~696,000 km), we can convert to an actual planet size in kilometers.

In [ ]:
# Step 8a — Measure the depth of the transit
# Take all data points close to transit center (within ±0.05 days)
in_transit_mask = (folded.time.value > -0.05) & (folded.time.value < 0.05)
brightness_during_transit = np.nanmedian(folded.flux.value[in_transit_mask])

# Depth = how much brightness dropped, as a percentage
depth_percent = (1 - brightness_during_transit) * 100

# Step 8b — Use depth to calculate planet/star size ratio
# Physics:  depth = (R_planet / R_star)²
# So:      R_planet / R_star = sqrt(depth)
radius_ratio = np.sqrt(depth_percent / 100)

# Step 8c — If star is Sun-sized, compute the planet's actual radius
SUN_RADIUS_KM    = 696_000     # km
JUPITER_RADIUS_KM = 69_911     # km
planet_radius_km = radius_ratio * SUN_RADIUS_KM
planet_radius_jupiters = planet_radius_km / JUPITER_RADIUS_KM

# Print the science!
print("=" * 60)
print("  📊 DISCOVERY REPORT — KEPLER-8b")
print("=" * 60)
print(f"  Measured transit depth:     {depth_percent:.3f} %")
print(f"  Planet/star radius ratio:   {radius_ratio:.4f}")
print(f"  Estimated planet radius:    {planet_radius_km:,.0f} km")
print(f"                              = {planet_radius_jupiters:.2f} × Jupiter")
print(f"  Orbital period (year):      {PERIOD:.4f} Earth days")
print(f"  Planet type:                Hot Jupiter ✓")
print("=" * 60)
print()
print("  🎉 You just discovered a planet 1,160 light-years away,")
print("     using the same data and same tools NASA scientists use!")

---
## 🎉 What You Just Did

In 8 steps you:

1. ✅ Downloaded real data from the Kepler Space Telescope  
2. ✅ Plotted 3 months of brightness measurements  
3. ✅ Cleaned the data to highlight the transit signal  
4. ✅ Folded every transit into one clean dip  
5. ✅ Measured the transit depth from real data  
6. ✅ Calculated the planet's actual size  

**This is exactly how every confirmed exoplanet is verified.** NASA scientists do this same workflow every day.

### Want to try another planet?

We've prepared an Excel file with parameters for **4 other planets** you can try. Just change the target name, period, and T0 in this notebook and run again.

| Planet | Period (days) | What's special |
|---|---|---|
| HAT-P-7 | 2.20 | Very deep transit, bright star |
| TrES-2 | 2.47 | Deepest transit (~1.5%) |
| Kepler-7 | 4.89 | A "puffy" Hot Jupiter |
| Kepler-10 | 0.84 | Rocky planet — TINY dip |

Open the Excel file for exact parameters, then come back and change `"Kepler-8"`, `PERIOD`, and `T0` in this notebook.

🔭 Keep looking up.